In [36]:
import os
import pdfplumber
from pdf2image import convert_from_path
import pytesseract
import cv2
from PIL import Image, ImageOps
import numpy as np

def extract_paragraphs_from_pdf(pdf_path):
    paragraphs = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages[11:17], start=1):
            text = page.extract_text()
            if text:
                page_paragraphs = text.split('.\n')
                page_paragraphs = [para.strip() for para in page_paragraphs if para.strip()]
                for para in page_paragraphs:
                    paragraphs.append({'page': page_number, 'text': para})
    
    return paragraphs

# Usage
pdf_path = "SSL.pdf"
paragraphs = extract_paragraphs_from_pdf(pdf_path)

# # Print for inspection
# for i, para in enumerate(paragraphs[:5], start=1):
#     print(f"Paragraph {i} (Page {para['page']}):")
#     print(para['text'])
#     print("-" * 80)
paragraphs[5]

{'page': 2,
 'text': '2 INTRODUCTION\npredictors, both for actual prediction with future data and also to discover\nwhich predictors play an important role. A linear regression model assumes\nthat\np\nX\ny =β + x β +e , (1.1)\ni 0 ij j i\nj=1\nwhere β and β = (β ,β ,...β ) are unknown parameters and e is an error\n0 1 2 p i\nterm. The method of least squares provides estimates of the parameters by\nminimization of the least-squares objective function\nN p\nX X\nminimize (y β x β )2. (1.2)\ni 0 ij j\nβ0,β\ni=1\n− −\nj=1\nTypically all of the least-squares estimates from (1.2) will be nonzero. This\nwill make interpretation of the final model challenging if p is large. In fact, if\np > N, the least-squares estimates are not unique. There is an infinite set of\nsolutions that make the objective function equal to zero, and these solutions\nalmost surely overfit the data as well'}

In [34]:
def is_formula(text):
    """
    Heuristically decide if a given text block is a formula.
    This is a simple heuristic that checks for the presence of math symbols
    or a high ratio of non-alphanumeric characters.
    """
    math_symbols = set("=+-*/∑∫√πθ^")
    symbol_count = sum(1 for c in text if c in math_symbols)
    non_alpha_count = sum(1 for c in text if not c.isalnum() and not c.isspace())
    if len(text) == 0:
        return False
    # If more than 20% of characters are math symbols or over 30% are non-alphanumeric, consider it a formula.
    if (symbol_count / len(text) > 0.2) or (non_alpha_count / len(text) > 0.3):
        return True
    return False

def math_ocr(image):
    """
    Stub for a math OCR function.
    In production, you might replace this with a call to an API like Mathpix that returns LaTeX.
    Here we use pytesseract as a placeholder, which may not output perfect LaTeX.
    """
    # Preprocess image if needed (e.g., invert colors, sharpen)
    image = ImageOps.grayscale(image)
    # Use Tesseract to extract string (this may need heavy tweaking for formulas)
    ocr_result = pytesseract.image_to_string(image)
    # A simple post-processing: strip extra whitespace
    latex = ocr_result.strip()
    return latex

def process_page(image):
    """
    Process a single page image.
    Uses pytesseract to extract data with bounding boxes,
    groups words into blocks, and determines if a block is text or a formula.
    Returns a list of dictionaries with type, content, and bounding box info.
    """
    # Convert PIL image to OpenCV image for possible processing
    open_cv_image = np.array(image.convert('RGB'))
    # (Optional) Preprocess: convert to grayscale, apply thresholding if necessary.
    gray = cv2.cvtColor(open_cv_image, cv2.COLOR_RGB2GRAY)
    # You could add additional pre-processing here if needed

    # Use pytesseract to get detailed data (words with bounding boxes)
    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
    
    blocks = {}
    # Group words by block and paragraph
    for i in range(len(data['text'])):
        word = data['text'][i].strip()
        if not word:
            continue
        # Use a tuple (block_num, par_num) as a key
        key = (data['block_num'][i], data['par_num'][i])
        if key not in blocks:
            blocks[key] = {
                "words": [word],
                "left": data['left'][i],
                "top": data['top'][i],
                "right": data['left'][i] + data['width'][i],
                "bottom": data['top'][i] + data['height'][i]
            }
        else:
            blocks[key]["words"].append(word)
            # Update bounding box
            blocks[key]["left"] = min(blocks[key]["left"], data['left'][i])
            blocks[key]["top"] = min(blocks[key]["top"], data['top'][i])
            blocks[key]["right"] = max(blocks[key]["right"], data['left'][i] + data['width'][i])
            blocks[key]["bottom"] = max(blocks[key]["bottom"], data['top'][i] + data['height'][i])
    
    paragraphs = []
    # Process each block to classify and extract its content
    for key, block in blocks.items():
        block_text = " ".join(block["words"])
        # Use heuristic to decide if this block is a formula
        if is_formula(block_text):
            # Crop the block from the image using its bounding box
            box = (block["left"], block["top"], block["right"], block["bottom"])
            block_img = image.crop(box)
            latex = math_ocr(block_img)
            paragraphs.append({
                "type": "formula",
                "content": latex,
                "bbox": box
            })
        else:
            paragraphs.append({
                "type": "text",
                "content": block_text,
                "bbox": (block["left"], block["top"], block["right"], block["bottom"])
            })
    return paragraphs

def process_pdf(pdf_path):
    """
    Convert the PDF to images and process each page.
    Returns a list of paragraphs with their type (text/formula), content, and page number.
    """
    # Convert PDF pages to images
    pages = convert_from_path(pdf_path)
    all_paragraphs = []
    for page_num, page_image in enumerate(pages[11:17], start=1):
        print(f"Processing page {page_num}...")
        paragraphs = process_page(page_image)
        for para in paragraphs:
            para["page"] = page_num
            all_paragraphs.append(para)
    return all_paragraphs

# --- MAIN EXECUTION ---


pdf_path = "SSL.pdf"  # Replace with your PDF file path
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF file not found: {pdf_path}")

extracted_data = process_pdf(pdf_path)

# Print a summary of the extracted paragraphs
for i, para in enumerate(extracted_data, start=1):
    print(f"Paragraph {i} (Page {para['page']}, Type: {para['type']}):")
    print(para['content'])
    print("-" * 80)

PDFInfoNotInstalledError: Unable to get page count. Is poppler installed and in PATH?